In [0]:
%sql
CREATE VOLUME IF NOT EXISTS telecom.landing.raw_files;

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS telecom.bronze.checkpoint;


In [0]:
from pyspark.sql import functions as F
landing_path = "/Volumes/telecom/landing/raw_files/sip/"
checkpoint   = "/Volumes/telecom/bronze/checkpoints/sip_packets"

raw = (spark.readStream
       .format("cloudFiles")
       .option("cloudFiles.format", "csv")
       .option("header", "true")
       .option("cloudFiles.schemaLocation", checkpoint)
       .load(landing_path))

bronze = raw.select(
    F.col("`frame.number`").alias("frame_number"),
    F.col("`frame.time_epoch`").alias("time_epoch"),
    F.col("`ip.src`").alias("src_ip"),
    F.col("`ip.dst`").alias("dst_ip"),
    F.col("`sip.Call-ID`").alias("call_id"),
    F.col("`sip.CSeq`").alias("cseq"),
    F.col("`sip.Method`").alias("method"),
    F.col("`sip.Status-Code`").alias("status_code"),
    F.col("_metadata.file_path").alias("source_file"),
    F.current_timestamp().alias("ingested_at"),
)

(bronze.writeStream
       .option("checkpointLocation", checkpoint)
       .trigger(availableNow=True)
       .toTable("telecom.bronze.sip_packets"))


In [0]:
%sql
CREATE VOLUME IF NOT EXISTS telecom.bronze.checkpoints;

In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS telecom.bronze.checkpoints")

In [0]:
display(spark.table("telecom.bronze.sip_packets"))

In [0]:
spark.table("telecom.bronze.sip_packets").count()